# Camino standoff con 7 gases: atenuación por gas y envolvente

El caso de `run_simulation()` de FORWARD (`final_forward_attenuation.py`) descrito **entero en un YAML** ([`examples/standoff_7gases.yaml`](../examples/standoff_7gases.yaml)) y corrido con `run_path()`: H2O, CO2, O3, N2O, CO, CH4 y O2 con todos sus isotopólogos, camino de 1 m a 1013 hPa y 288.2 K, 600–2500 cm⁻¹ con paso 0.01 cm⁻¹, promediado en bines de 0.02 µm.

**Continuo:** el YAML activa el continuo MT_CKD de H2O (self + foreign, 620–2780 cm⁻¹) y de CO2 (por debajo de 2000 cm⁻¹), portado de photonix (`Forward_repo`). No hay continuo de O2/N2 (absorción inducida por colisiones). Para ver su efecto, el cálculo se repite sin continuo.

**Salidas:** atenuación en dB/m, `A = −10/L · log10(T)`, de cada gas por separado en los bines de 0.02 µm, y la **envolvente**: la suma de las atenuaciones de los 7 gases (el "Total" de FORWARD, `A_dbm_lam_sum`).

| FORWARD (`run_simulation`) | pylblrtm |
|---|---|
| `s.Pmol` (presión parcial, atm) | `path.gases` con `gas_units: atm` |
| `delta_um=0.020` | `output.bin_um: 0.02` |
| `res['lambda_centers']` | `res.wavelength_um` |
| `res['T_prod_samp']` | `res.transmittance_binned` |
| `A_dbm_lam_each[k]` | `res.attenuation_per_gas_db_m["CO2"]` (por nombre) |
| `A_dbm_lam_sum` | `sum(res.attenuation_per_gas_db_m.values())` |

En cvail: `sbatch scripts/run_notebook.sh notebooks/standoff_7gases.ipynb`. Requiere `pip install -e ".[notebook]"` (y `.[fast]` para el backend rápido).

In [ ]:
import contextlib
import copy
import importlib.util
import io
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from pylblrtm import load_config, run_path

# Raíz del repo, abriendo el notebook desde ahí o desde notebooks/
REPO = next(p for p in (Path.cwd(), Path.cwd().parent) if (p / "examples").is_dir())
YAML = REPO / "examples" / "standoff_7gases.yaml"
print(YAML.read_text(encoding="utf-8"))

In [ ]:
cfg = load_config(YAML)
cfg["output"]["save"]["path"] = str(REPO / "OUT")      # CSV en <repo>/OUT/
if importlib.util.find_spec("numba") is not None:       # Humlicek: mucho más rápido, < 1e-4 de wofz
    cfg["calculation"]["voigt"] = "humlicek"

def correr(config):
    t0 = time.perf_counter()
    with contextlib.redirect_stdout(io.StringIO()):      # HAPI imprime el progreso de cada descarga
        r = run_path(config)
    return r, time.perf_counter() - t0

res, t_con = correr(cfg)
sin = copy.deepcopy(cfg)
sin["calculation"]["continuum"] = False
sin["output"]["save"]["enabled"] = False
res_lineas, t_sin = correr(sin)

print(res)
print(f"con continuo {t_con:.0f} s · sin continuo {t_sin:.0f} s · voigt={cfg['calculation']['voigt']}")
print(f"gases sin líneas en el rango: {res.skipped_gases or 'ninguno'}")

lam = res.wavelength_um                                  # lambda_centers
A_gas = res.attenuation_per_gas_db_m                     # {gas: dB/m por bin}
A_env = sum(A_gas.values())                              # envolvente (A_dbm_lam_sum de FORWARD)
A_gas_lineas = res_lineas.attenuation_per_gas_db_m
print(f"{lam.size} bines de {cfg['output']['bin_um']} µm entre {lam[0]:.3f} y {lam[-1]:.3f} µm")

## Tabla: atenuación por gas y envolvente

In [ ]:
ventana = (lam >= 8.0) & (lam <= 12.0)
print(f"{'Gas':<10}{'A máx [dB/m]':>14}{'en λ [µm]':>11}{'A media 8–12 µm':>18}{'  … sin continuo':>18}")
print("─" * 71)
filas = list(A_gas.items()) + [("ENVOLVENTE", A_env)]
for gas, A in filas:
    k = int(np.argmax(A))
    sin_c = A_gas_lineas.get(gas, sum(A_gas_lineas.values()) if gas == "ENVOLVENTE" else None)
    extra = f"{np.mean(sin_c[ventana]):18.3e}" if gas in ("H2O", "CO2", "ENVOLVENTE") else f"{'(igual)':>18}"
    print(f"{gas:<10}{A[k]:14.3e}{lam[k]:11.3f}{np.mean(A[ventana]):18.3e}{extra}")

## Atenuación de cada gas y envolvente

In [ ]:
SURFACE, INK, INK_2, MUTED, GRID, AXIS = "#fcfcfb", "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
# Paleta categórica en orden fijo (validada: CVD ΔE ≥ 9.1 entre vecinos); la envolvente en tinta
COLORES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6"]
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "axes.edgecolor": AXIS,
    "axes.labelcolor": INK_2, "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False, "font.size": 10,
    "legend.frameon": False, "legend.labelcolor": INK_2,
})
PISO = 1e-8   # dB/m: los bines sin absorción se dibujan en el piso del eje log

fig, ax = plt.subplots(figsize=(12, 5))
for (gas, A), color in zip(A_gas.items(), COLORES):
    ax.semilogy(lam, np.clip(A, PISO, None), color=color, linewidth=1.2, label=gas)
ax.semilogy(lam, np.clip(A_env, PISO, None), color=INK, linewidth=2.0, label="Envolvente")
k = int(np.argmax(A_env))
ax.annotate(f"máx {A_env[k]:.2e} dB/m\n{lam[k]:.2f} µm", xy=(lam[k], A_env[k]), xytext=(10, -4),
            textcoords="offset points", color=INK_2, fontsize=9)
ax.set_ylim(PISO, A_env.max() * 5)
ax.set_xlabel("Longitud de onda [µm]")
ax.set_ylabel("Atenuación [dB/m]")
ax.set_title(f"Atenuación por gas y envolvente · bines de {cfg['output']['bin_um']} µm · "
             f"L = {cfg['path']['length_m']} m, {cfg['path']['pressure_hpa']} hPa, "
             f"{cfg['path']['temperature_k']} K, con continuo", color=INK, loc="left")
ax.legend(ncol=8, loc="upper center", bbox_to_anchor=(0.5, -0.13))
fig.tight_layout()
plt.show()

In [ ]:
# Un panel por gas, misma escala: cada uno se lee sin que lo tapen los demás
paneles = list(A_gas.items()) + [("Envolvente", A_env)]
fig, axes = plt.subplots(len(paneles), 1, figsize=(11, 1.7 * len(paneles)), sharex=True, sharey=True)
for ax, (nombre, A) in zip(axes, paneles):
    ax.semilogy(lam, np.clip(A, PISO, None), color=INK if nombre == "Envolvente" else "#2a78d6", linewidth=1.2)
    ax.text(0.005, 0.80, f"{nombre} · máx {A.max():.2e} dB/m", transform=ax.transAxes,
            color=INK, fontsize=10, fontweight="bold")
axes[0].set_ylim(PISO, A_env.max() * 5)
axes[-1].set_xlabel("Longitud de onda [µm]")
fig.supylabel("Atenuación [dB/m]", color=INK_2)
fig.tight_layout()
plt.show()

## Efecto del continuo (H2O y CO2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.6), sharey=True)
for ax, gas in zip(axes, ["H2O", "CO2"]):
    ax.semilogy(lam, np.clip(A_gas_lineas[gas], PISO, None), color="#2a78d6", linewidth=1.2, label="solo líneas")
    ax.semilogy(lam, np.clip(A_gas[gas], PISO, None), color="#eb6834", linewidth=1.2, label="líneas + continuo")
    ax.set_title(gas, color=INK, loc="left")
    ax.set_xlabel("Longitud de onda [µm]")
    ax.legend(loc="lower left")
axes[0].set_ylabel("Atenuación [dB/m]")
axes[0].set_ylim(PISO, A_env.max() * 5)
fig.tight_layout()
plt.show()

print("Archivos guardados por output.save:", sorted(p.name for p in (REPO / "OUT").glob("standoff_7gases*")))